# 03 — RFM Analysis

## Objective
Engineer Recency, Frequency, and Monetary features from the cleaned 
transaction data, score each customer across all three dimensions, 
and assign meaningful segments to drive business decisions.

## Contents
1. Load cleaned data
2. Feature engineering — Recency, Frequency, Monetary
3. RFM scoring
4. Customer segmentation
5. Segment analysis

---

In [1]:
import datetime as dt
import pandas as pd

df = pd.read_csv('../data/processed/df_clean.csv')

In [2]:
print(df.shape)
df.head()

(392693, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Monetary Calculation

In [3]:
df['Total'] = df['Quantity'] * df['Price']
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [4]:
rfm = df[['Customer ID', 'Total']].groupby('Customer ID').sum()

In [5]:
rfm= rfm.reset_index()
rfm.columns = ['Customer ID', 'Monetary']

In [6]:
rfm.head()

,Customer ID,Monetary
0,12346.0,77183.60
1,12347.0,4310.00
2,12348.0,1797.24
3,12349.0,1757.55
4,12350.0,334.40


### Frequency Calculation

In [ ]:
rfm = rfm.merge(df[['Customer ID','Invoice']].groupby('Customer ID').nunique().reset_index(), how = 'inner')

In [17]:
rfm = rfm.rename(columns={'Invoice':'Frequency'})
rfm.head()

,Customer ID,Monetary,Frequency
0,12346.0,77183.60,1
1,12347.0,4310.00,7
2,12348.0,1797.24,4
3,12349.0,1757.55,1
4,12350.0,334.40,1


### Recency Calculation

In [31]:
df['InvoiceDate'].dtype

<StringDtype(storage='python', na_value=nan)>

In [32]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [39]:
print(df['InvoiceDate'].max())
ref_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

2011-12-09 12:50:00


We can see that the last invoice date was on 09-12-2011. We will use the day after the last invoice date as the reference date to calculate the recency column.

In [55]:
recency = df[['Customer ID','InvoiceDate']].groupby('Customer ID').max().reset_index()
recency.head()

,Customer ID,InvoiceDate
0,12346.0,2011-01-18 10:01:00
1,12347.0,2011-12-07 15:52:00
2,12348.0,2011-09-25 13:13:00
3,12349.0,2011-11-21 09:51:00
4,12350.0,2011-02-02 16:01:00


In [56]:
recency['Recency'] = (ref_date - recency['InvoiceDate']).dt.days
recency.head()

,Customer ID,InvoiceDate,Recency
0,12346.0,2011-01-18 10:01:00,326
1,12347.0,2011-12-07 15:52:00,2
2,12348.0,2011-09-25 13:13:00,75
3,12349.0,2011-11-21 09:51:00,19
4,12350.0,2011-02-02 16:01:00,310


In [57]:
recency = recency.drop(['InvoiceDate'], axis = 1)
recency.head()

,Customer ID,Recency
0,12346.0,326
1,12347.0,2
2,12348.0,75
3,12349.0,19
4,12350.0,310


In [60]:
rfm = rfm.merge(recency, how='inner')

In [62]:
rfm.head()

,Customer ID,Monetary,Frequency,Recency
0,12346.0,77183.60,1,326
1,12347.0,4310.00,7,2
2,12348.0,1797.24,4,75
3,12349.0,1757.55,1,19
4,12350.0,334.40,1,310


Double Checking rfm table for valid values

In [65]:
df['Customer ID'].nunique()

4338

In [64]:
print(rfm.shape)
rfm.describe()

(4338, 4)


,Customer ID,Monetary,Frequency,Recency
count,4338.000000,4338.000000,4338.000000,4338.000000
mean,15300.408022,2048.692230,4.272015,92.536422
std,1721.808492,8985.229676,7.697998,100.014169
min,12346.000000,3.750000,1.000000,1.000000
25%,13813.250000,306.482500,1.000000,18.000000
50%,15299.500000,668.570000,2.000000,51.000000
75%,16778.750000,1660.597500,5.000000,142.000000
max,18287.000000,280206.020000,209.000000,374.000000
